In [1]:
import pickle
import pandas as pd
from pathlib import Path

# ============================================================
# STEP A — Validate TCGA genes against Geneformer vocabulary
# ============================================================

# ---------- 1. CHANGE THIS PATH ----------
Train_DATA_FILE = Path(r"F:\GeneformerProject\data\preprocessed_5_cancers\\train.csv")
Test_DATA_FILE = Path(r"F:\GeneformerProject\data\preprocessed_5_cancers\\test.csv")
Valid_DATA_FILE = Path(r"F:\GeneformerProject\data\preprocessed_5_cancers\\validation.csv")

# ---------- 2. Load your TCGA data ----------
print("Loading TCGA data...")

if Train_DATA_FILE.suffix.lower() == ".csv":
    train = pd.read_csv(Train_DATA_FILE)
    test = pd.read_csv(Test_DATA_FILE)
    val = pd.read_csv(Valid_DATA_FILE)

elif Train_DATA_FILE.suffix.lower() in [".pkl", ".pickle"]:
   train = pd.read_pickle(Train_DATA_FILE)
   test = pd.read_pickle(Test_DATA_FILE)
   val = pd.read_pickle(Valid_DATA_FILE)

elif Train_DATA_FILE.suffix.lower() == ".parquet":
   train = pd.read_parquet(Train_DATA_FILE)
   test = pd.read_parquet(Test_DATA_FILE)
   val = pd.read_parquet(Valid_DATA_FILE)

else:
    raise ValueError(f"Unsupported file type: {DATA_FILE.suffix}")

print("Train:",  train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print("\nColumns:")
print("Train:", len( train.columns))


# 3. Identify gene columns
# ============================================================
LABEL_COLUMNS = ["Cancer_Type", "tissue_type"]

GENE_COLUMNS = [
    col for col in train.columns
    if col not in LABEL_COLUMNS
]

print("Number of expression columns:", len(GENE_COLUMNS))
print("Label columns:", LABEL_COLUMNS)

print("\nFirst 20 genes:")
print(GENE_COLUMNS[:20])

print("\nLast 10 genes:")
print(GENE_COLUMNS[-10:])


Loading TCGA data...
Train: (2411, 20533)
Validation: (517, 20533)
Test: (517, 20533)

Columns:
Train: 20533
Number of expression columns: 20531
Label columns: ['Cancer_Type', 'tissue_type']

First 20 genes:
['100130426', '100133144', '100134869', '10357', '10431', '136542', '155060', '26823', '280660', '317712', '340602', '388795', '390284', '391343', '391714', '404770', '441362', '442388', '553137', '57714']

Last 10 genes:
['ZWILCH', 'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B', 'ZYX', 'ZZEF1', 'ZZZ3']


In [2]:

# 4. Load Geneformer token dictionary 
# ============================================================
# Try the standard location first

TOKEN_DICT_PATH = Path(r"F:\GeneformerProject\Geneformer\geneformer\token_dictionary_gc104M.pkl")

with open(TOKEN_DICT_PATH, "rb") as f:
    token_dictionary = pickle.load(f)

SPECIAL_TOKENS = {
    "<pad>",
    "<mask>",
    "<cls>",
    "<eos>"
}

geneformer_ensembl = {
    str(gene)
    for gene in token_dictionary
    if str(gene) not in SPECIAL_TOKENS
}

print("Total token dictionary entries:", len(token_dictionary))
print("Geneformer gene entries:", len(geneformer_ensembl))

print("\nFirst 10 genes:")
print(list(geneformer_ensembl)[:10])

# 5. IDENTIFIER TYPES
# ============================================================

gene_strings = [str(g) for g in GENE_COLUMNS]

ensembl_ids = [
    g for g in gene_strings
    if g.startswith("ENSG")
]

entrez_ids = [
    g for g in gene_strings
    if g.isdigit()
]

gene_symbols = [
    g for g in gene_strings
    if not g.startswith("ENSG") and not g.isdigit()
]

print("Identifier summary")
print("-" * 40)
print("Total:", len(gene_strings))
print("Ensembl:", len(ensembl_ids))
print("Entrez-like:", len(entrez_ids))
print("Gene symbols:", len(gene_symbols))

Total token dictionary entries: 20275
Geneformer gene entries: 20271

First 10 genes:
['ENSG00000230778', 'ENSG00000144741', 'ENSG00000267795', 'ENSG00000124226', 'ENSG00000171794', 'ENSG00000065427', 'ENSG00000075292', 'ENSG00000185947', 'ENSG00000178440', 'ENSG00000171204']
Identifier summary
----------------------------------------
Total: 20531
Ensembl: 0
Entrez-like: 29
Gene symbols: 20502


In [3]:
# 6. GENE ANNOTATION
# ============================================================
import mygene

mg = mygene.MyGeneInfo()

# 7A. Symbol → Ensembl
# ============================================================

symbol_results = mg.querymany(
    gene_symbols,
    scopes="symbol",
    fields="ensembl.gene,symbol",
    species="human",
    as_dataframe=False,
    verbose=False
)

print("Input symbols:", len(gene_symbols))
print("Returned records:", len(symbol_results))

# 7B. Entrez → Ensembl
# ============================================================

entrez_results = mg.querymany(
    entrez_ids,
    scopes="entrezgene",
    fields="ensembl.gene,symbol",
    species="human",
    as_dataframe=False,
    verbose=False
)

print("Input Entrez IDs:", len(entrez_ids))
print("Returned records:", len(entrez_results))

# 7C. CHECK MULTIPLE RESULTS
# ============================================================

from collections import Counter

symbol_counts = Counter(
    r.get("query")
    for r in symbol_results
    if "query" in r
)

multiple_symbols = {
    gene: count
    for gene, count in symbol_counts.items()
    if count > 1
}

print("Unique queried symbols:", len(symbol_counts))
print("Symbols with multiple returned records:", len(multiple_symbols))

print("\nExamples:")
for gene, count in list(multiple_symbols.items())[:20]:
    print(gene, "->", count, "records")

print("="*60)
no_result = [
    r for r in symbol_results
    if r.get("notfound", False)
]

print("Symbols with no result:", len(no_result))

print("="*60)
no_ensembl = [
    r for r in symbol_results
    if not r.get("ensembl")
]

print("Results without Ensembl:", len(no_ensembl))

print("="*60)
multiple_ensembl = []

for r in symbol_results:
    ens = r.get("ensembl")

    if isinstance(ens, dict):
        gene = ens.get("gene")

        if isinstance(gene, list) and len(gene) > 1:
            multiple_ensembl.append(r)

print("Records with multiple Ensembl IDs:",
      len(multiple_ensembl))


Input symbols: 20502
Returned records: 20882
Input Entrez IDs: 29
Returned records: 29
Unique queried symbols: 20502
Symbols with multiple returned records: 291

Examples:
ABCA11P -> 2 records
ABCA17P -> 2 records
ABCC13 -> 2 records
ABCC6P1 -> 2 records
ABCC6P2 -> 3 records
ADAM21P1 -> 2 records
ADAM3A -> 2 records
ADAM6 -> 3 records
AGAP11 -> 2 records
ALOX12P2 -> 2 records
AMZ2P1 -> 2 records
ANKRD26P1 -> 2 records
ANKRD36BP1 -> 2 records
ANXA2P1 -> 2 records
ANXA2P2 -> 2 records
ANXA2P3 -> 2 records
APOC1P1 -> 2 records
AQP7P1 -> 2 records
ARGFXP2 -> 2 records
ATP6AP1L -> 2 records
Symbols with no result: 3287
Results without Ensembl: 3517
Records with multiple Ensembl IDs: 0


In [4]:
# 8A. BUILD ONE-TO-ONE SYMBOL → ENSEMBL MAPPING
# ============================================================
def get_ensembl_ids(result):
    """
    Return all Ensembl gene IDs from one MyGene result.
    """

    ens = result.get("ensembl")

    if ens is None:
        return []

    if isinstance(ens, dict):
        gene = ens.get("gene")

        if isinstance(gene, str):
            return [gene]

        if isinstance(gene, list):
            return [
                g for g in gene
                if isinstance(g, str)
            ]

    elif isinstance(ens, list):
        ids = []

        for item in ens:
            if isinstance(item, dict):
                gene = item.get("gene")

                if isinstance(gene, str):
                    ids.append(gene)

        return ids

    return []



# ------------------------------------------------------------
#8B. Symbol → Ensembl
# ------------------------------------------------------------
from collections import defaultdict

symbol_to_ensembl = defaultdict(set)

for result in symbol_results:

    query = str(result.get("query"))

    for ensembl_id in get_ensembl_ids(result):

        symbol_to_ensembl[query].add(ensembl_id)


# ------------------------------------------------------------
# Classify symbol mappings
# ------------------------------------------------------------

unique_symbol_mapping = {}
ambiguous_symbol_mapping = {}
unmapped_symbols = []

for symbol in gene_symbols:

    matches = symbol_to_ensembl.get(symbol, set())

    if len(matches) == 1:

        unique_symbol_mapping[symbol] = next(iter(matches))

    elif len(matches) > 1:

        ambiguous_symbol_mapping[symbol] = sorted(matches)

    else:

        unmapped_symbols.append(symbol)

# ------------------------------------------------------------
# 8C. Entrez → Ensembl
# ------------------------------------------------------------

entrez_to_ensembl = {}

unmapped_entrez = []

for entrez in entrez_ids:

    matches = set()

    for result in entrez_results:

        if str(result.get("query")) == str(entrez):

            matches.update(
                get_ensembl_ids(result)
            )

    if len(matches) == 1:

        entrez_to_ensembl[entrez] = next(iter(matches))

    else:

        unmapped_entrez.append(entrez)        

In [5]:
# 9. COMBINE ALL IDENTIFIER TYPES
# ============================================================

gene_to_ensembl = {}

# ------------------------------------------------------------
# Existing Ensembl IDs
# ------------------------------------------------------------

for gene in ensembl_ids:

    gene_to_ensembl[gene] = gene


# ------------------------------------------------------------
# Gene symbols
# ------------------------------------------------------------

gene_to_ensembl.update(
    unique_symbol_mapping
)


# ------------------------------------------------------------
# Entrez IDs
# ------------------------------------------------------------

gene_to_ensembl.update(
    entrez_to_ensembl
)


print("=" * 60)
print("FINAL IDENTIFIER → ENSEMBL MAPPING")
print("=" * 60)

print("Original genes:", len(gene_strings))
print("Mapped genes:", len(gene_to_ensembl))

# 10. CREATE MAPPING QUALITY TABLE
# ============================================================

mapping_rows = []

for original_id in gene_strings:

    ensembl_id = gene_to_ensembl.get(original_id)

    if ensembl_id is None:

        status = "unmapped"

    elif ensembl_id in geneformer_ensembl:

        status = "geneformer"

    else:

        status = "not_in_geneformer"

    mapping_rows.append({
        "original_id": original_id,
        "ensembl_id": ensembl_id,
        "status": status
    })


mapping_df = pd.DataFrame(mapping_rows)

print("=" * 60)
print("FINAL IDENTIFIER → ENSEMBL MAPPING")
print("=" * 60)
print(mapping_df.head(20))

print("\nStatus:")
print(mapping_df["status"].value_counts())

FINAL IDENTIFIER → ENSEMBL MAPPING
Original genes: 20531
Mapped genes: 15705
FINAL IDENTIFIER → ENSEMBL MAPPING
   original_id       ensembl_id             status
0    100130426             None           unmapped
1    100133144             None           unmapped
2    100134869  ENSG00000290945  not_in_geneformer
3        10357             None           unmapped
4        10431             None           unmapped
5       136542             None           unmapped
6       155060  ENSG00000290600  not_in_geneformer
7        26823  ENSG00000201659  not_in_geneformer
8       280660  ENSG00000290708  not_in_geneformer
9       317712             None           unmapped
10      340602  ENSG00000187690         geneformer
11      388795  ENSG00000215529         geneformer
12      390284             None           unmapped
13      391343             None           unmapped
14      391714  ENSG00000250374         geneformer
15      404770  ENSG00000231649  not_in_geneformer
16      441362       

In [6]:
# 11. GENEFORMER COMPATIBILITY
# ============================================================

geneformer_genes = mapping_df[
    mapping_df["status"] == "geneformer"
].copy()

not_in_geneformer = mapping_df[
    mapping_df["status"] == "not_in_geneformer"
].copy()

unmapped = mapping_df[
    mapping_df["status"] == "unmapped"
].copy()


print("=" * 60)
print("GENEFORMER COMPATIBILITY")
print("=" * 60)

print("Original genes:           ", len(mapping_df))
print("Geneformer-compatible:    ", len(geneformer_genes))
print("Not in Geneformer:        ", len(not_in_geneformer))
print("Unmapped:                  ", len(unmapped))

# 12. CHECK DUPLICATE ENSEMBL IDS
# ============================================================

valid_mapping = mapping_df[
    mapping_df["ensembl_id"].notna()
].copy()

duplicate_counts = (
    valid_mapping["ensembl_id"]
    .value_counts()
)

duplicate_counts = duplicate_counts[
    duplicate_counts > 1
]

print("=" * 60)
print("DUPLICATE ENSEMBL IDs")
print("=" * 60)

print(
    "Ensembl IDs with multiple original identifiers:",
    len(duplicate_counts)
)

if len(duplicate_counts) > 0:
    print("\nExamples:")
    print(duplicate_counts.head(20))

GENEFORMER COMPATIBILITY
Original genes:            20531
Geneformer-compatible:     15100
Not in Geneformer:         605
Unmapped:                   4826
DUPLICATE ENSEMBL IDs
Ensembl IDs with multiple original identifiers: 0


In [7]:
# 13. SAVE MAPPING RESULTS
# ============================================================
import os
DATA_DIR = "F:\GeneformerProject\data\preprocessed_5_cancers"

MAPPING_PATH = os.path.join(
    DATA_DIR,
    "geneformer_gene_mapping.csv"
)

UNMAPPED_PATH = os.path.join(
    DATA_DIR,
    "geneformer_unmapped_genes.csv"
)

AMBIGUOUS_PATH = os.path.join(
    DATA_DIR,
    "geneformer_ambiguous_symbols.csv"
)


# Full mapping
mapping_df.to_csv(
    MAPPING_PATH,
    index=False
)


# Unmapped
unmapped.to_csv(
    UNMAPPED_PATH,
    index=False
)


# Ambiguous
ambiguous_df = pd.DataFrame([
    {
        "symbol": symbol,
        "ensembl_ids": ";".join(ids)
    }
    for symbol, ids in ambiguous_symbol_mapping.items()
])

ambiguous_df.to_csv(
    AMBIGUOUS_PATH,
    index=False
)


print("=" * 60)
print("FILES SAVED")
print("=" * 60)

print(MAPPING_PATH)
print(UNMAPPED_PATH)
print(AMBIGUOUS_PATH)

FILES SAVED
F:\GeneformerProject\data\preprocessed_5_cancers\geneformer_gene_mapping.csv
F:\GeneformerProject\data\preprocessed_5_cancers\geneformer_unmapped_genes.csv
F:\GeneformerProject\data\preprocessed_5_cancers\geneformer_ambiguous_symbols.csv


In [8]:
# 14. FINAL CHECK
# ============================================================

print("=" * 60)
print("FINAL CHECK")
print("=" * 60)

original = len(mapping_df)

mapped = mapping_df["ensembl_id"].notna().sum()

geneformer = (
    mapping_df["status"] == "geneformer"
).sum()

not_geneformer = (
    mapping_df["status"] == "not_in_geneformer"
).sum()

unmapped_count = (
    mapping_df["status"] == "unmapped"
).sum()


print(f"Original genes:          {original}")
print(f"Mapped to Ensembl:       {mapped}")
print(f"Geneformer-compatible:   {geneformer}")
print(f"Not in Geneformer:       {not_geneformer}")
print(f"Unmapped:                {unmapped_count}")

print()
print("Check:")

print(
    geneformer +
    not_geneformer +
    unmapped_count,
    "==",
    original
)

FINAL CHECK
Original genes:          20531
Mapped to Ensembl:       15705
Geneformer-compatible:   15100
Not in Geneformer:       605
Unmapped:                4826

Check:
20531 == 20531


In [9]:
# 15. CREATE FINAL GENEFORMER-READY DATASETS
# ============================================================

label_columns = [
    "Cancer_Type",
    "tissue_type"
]

# ------------------------------------------------------------
# 1. Geneformer-compatible mappings only
# ------------------------------------------------------------

geneformer_mapping = mapping_df[
    mapping_df["status"] == "geneformer"
].copy()

geneformer_mapping["original_id"] = (
    geneformer_mapping["original_id"].astype(str)
)

geneformer_mapping["ensembl_id"] = (
    geneformer_mapping["ensembl_id"].astype(str)
)

print("Geneformer-compatible genes:",
      len(geneformer_mapping))


# ------------------------------------------------------------
# 2. Create original → Ensembl dictionary
# ------------------------------------------------------------

rename_dict = dict(
    zip(
        geneformer_mapping["original_id"],
        geneformer_mapping["ensembl_id"]
    )
)


# ------------------------------------------------------------
# 3. IMPORTANT:
#    Keep the ORIGINAL IDs in the same order as mapping
# ------------------------------------------------------------

original_gene_ids = (
    geneformer_mapping["original_id"].tolist()
)

ensembl_gene_ids = (
    geneformer_mapping["ensembl_id"].tolist()
)


# ------------------------------------------------------------
# 4. Check that all original IDs exist in datasets
# ------------------------------------------------------------

missing_train = [
    g for g in original_gene_ids
    if g not in train.columns
]

missing_val = [
    g for g in original_gene_ids
    if g not in val.columns
]

missing_test = [
    g for g in original_gene_ids
    if g not in test.columns
]

print("=" * 60)
print("MISSING GENE CHECK")
print("=" * 60)

print("Train missing:", len(missing_train))
print("Validation missing:", len(missing_val))
print("Test missing:", len(missing_test))


# ------------------------------------------------------------
# 5. Select only Geneformer-compatible genes
# ------------------------------------------------------------

train_geneformer = train[
    original_gene_ids + label_columns
].copy()

val_geneformer = val[
    original_gene_ids + label_columns
].copy()

test_geneformer = test[
    original_gene_ids + label_columns
].copy()

print("train before rename:")
print(train_geneformer.columns)

print("val before rename:")
print(val_geneformer.columns)

print("test before rename:")
print(test_geneformer.columns)
# ------------------------------------------------------------
# 6. Rename original IDs → Ensembl IDs
# ------------------------------------------------------------

train_geneformer = train_geneformer.rename(
    columns=rename_dict
)

val_geneformer = val_geneformer.rename(
    columns=rename_dict
)

test_geneformer = test_geneformer.rename(
    columns=rename_dict
)
print("train after rename:")
print(train_geneformer.columns)

print("val after rename:")
print(val_geneformer.columns)

print("test after rename:")
print(test_geneformer.columns)

# ------------------------------------------------------------
# 7. Final column order
# ------------------------------------------------------------

train_geneformer = train_geneformer[
    ensembl_gene_ids + label_columns
]

val_geneformer = val_geneformer[
    ensembl_gene_ids + label_columns
]

test_geneformer = test_geneformer[
    ensembl_gene_ids + label_columns
]


# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 60)
print("FINAL GENEFORMER DATASETS")
print("=" * 60)

print("Train:", train_geneformer.shape)
print(train_geneformer.columns)
print("Validation:", val_geneformer.shape)
print("Test:", test_geneformer.shape)

print()
print("Gene columns:", len(ensembl_gene_ids))
print("Label columns:", len(label_columns))

print()
print("First 10 columns:")
print(train_geneformer.columns[:10].tolist())

print()
print("Last 5 columns:")
print(train_geneformer.columns[-5:].tolist())

Geneformer-compatible genes: 15100
MISSING GENE CHECK
Train missing: 0
Validation missing: 0
Test missing: 0
train before rename:
Index(['340602', '388795', '391714', '90288', 'A1BG', 'A1CF', 'A2M', 'A2ML1',
       'A4GALT', 'A4GNT',
       ...
       'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B', 'ZZEF1', 'ZZZ3',
       'Cancer_Type', 'tissue_type'],
      dtype='object', length=15102)
val before rename:
Index(['340602', '388795', '391714', '90288', 'A1BG', 'A1CF', 'A2M', 'A2ML1',
       'A4GALT', 'A4GNT',
       ...
       'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B', 'ZZEF1', 'ZZZ3',
       'Cancer_Type', 'tissue_type'],
      dtype='object', length=15102)
test before rename:
Index(['340602', '388795', '391714', '90288', 'A1BG', 'A1CF', 'A2M', 'A2ML1',
       'A4GALT', 'A4GNT',
       ...
       'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B', 'ZZEF1', 'ZZZ3',
       'Cancer_Type', 'tissue_type'],
      dtype='object', length=15102)
train after rename:
Index(['ENSG0

In [10]:
# 16. SAVE GENEFORMER-READY DATASETS
# ============================================================

output_dir = Path(
    "F:\GeneformerProject\data\geneformer_ready\preprocessed_5_cancers"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

# Save as Parquet
train_geneformer.to_parquet(
    output_dir / "train_geneformer.parquet",
    index=False
)

val_geneformer.to_parquet(
    output_dir / "validation_geneformer.parquet",
    index=False
)

test_geneformer.to_parquet(
    output_dir / "test_geneformer.parquet",
    index=False
)

print("=" * 60)
print("SAVED SUCCESSFULLY")
print("=" * 60)

print(output_dir)

SAVED SUCCESSFULLY
F:\GeneformerProject\data\geneformer_ready\preprocessed_5_cancers


In [11]:
#  17. Verify
# =============================================================
check = pd.read_parquet(
    r"F:\GeneformerProject\data\geneformer_ready\preprocessed_5_cancers\train_geneformer.parquet"
)

print(check.shape)
print(check.head())

# Check that all gene columns are valid Ensembl IDs
invalid_genes = [
    g for g in geneformer_genes
    if not str(g).startswith("ENSG")
]

print("Invalid gene IDs:", len(invalid_genes))

if invalid_genes:
    print(invalid_genes[:20])


(2411, 15102)
   ENSG00000187690  ENSG00000215529  ENSG00000250374  ENSG00000172771  \
0             1.03             0.60              0.0             4.91   
1             0.00             1.19              0.0             4.79   
2             0.00             2.05              0.0             1.63   
3             0.91             2.64              0.0             7.31   
4             0.00             2.75              0.0             1.89   

   ENSG00000121410  ENSG00000148584  ENSG00000175899  ENSG00000166535  \
0             4.88             1.03            13.64             1.03   
1             6.92             0.40            14.65             1.38   
2             4.94             8.82            13.45             0.00   
3             6.72             0.00            14.32             1.46   
4             6.44             0.38            16.38             4.66   

   ENSG00000128274  ENSG00000118017  ...  ENSG00000122952  ENSG00000198205  \
0             7.21            